# Ball Flight to Biomechanics Analysis

This notebook builds a biomechanics-to-ball-flight modeling pipeline by first standardizing the dataset for handedness and player size, ensuring all biomechanical variables are aligned to a consistent dominant-side reference frame. It then uses grouped Random Forest models to estimate optimal ball-flight ranges based on high-quality shots, and follows with LightGBM regression models that map biomechanical movement variables to key ball-flight outcomes. Model performance is evaluated using MAE, RMSE, and R², while gain-based feature importance identifies the most influential biomechanical drivers of each ball-flight metric, setting up the framework for SHAP-based interpretability and individualized player feedback.

#### 1. Load Data and Packages

In [28]:
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestRegressor

In [18]:
df = pd.read_csv('capstone2026v2.csv')
df.head()

,Name,Date,Shot.Type,Shot.Location,Shot.Distance,Shot.X.Pos,Shot.Y.Pos,Made,Ball_Depth,left_right,...,maxElbowExtensionLeftPostHitch,maxElbowExtensionRightPostHitch,maxElbowExtensionAvgPostHitch,timeToMaxElbowExtensionLeftPostHitch,timeToMaxElbowExtensionRightPostHitch,ElbowTotalROMRight,ElbowTotalROMLeft,ht,wt,hand
0,Player 1,5/27/2025 8:18,Off the Dribble,Left Corner Three,22.324880,-22.323844,-0.215156,False,22.61,-2.18,...,16.8562,22.3653,19.6107,0.650,0.650,-81.4055,-122.0768,76.0,86.0,Left
1,Player 1,5/27/2025 8:18,Off the Dribble,Left Corner Three,22.507735,-22.507038,-0.177118,True,12.96,-3.84,...,10.1352,23.7183,16.9268,0.567,0.559,-78.7364,-119.9664,76.0,86.0,Left
2,Player 1,5/27/2025 8:18,Off the Dribble,Left Corner Three,22.798792,-22.797181,-0.270969,True,14.14,-1.73,...,13.4128,21.6987,17.5557,0.559,0.550,-81.1395,-124.8665,76.0,86.0,Left
3,Player 1,5/27/2025 8:18,Off the Dribble,Left Corner Three,22.980568,-22.977321,-0.386303,False,6.61,-2.44,...,13.1956,24.0487,18.6222,0.525,0.517,-75.8957,-120.4512,76.0,86.0,Left
4,Player 1,5/27/2025 8:18,Off the Dribble,Left Corner Three,22.559167,-22.556121,-0.370699,True,15.83,-0.41,...,14.6284,22.8648,18.7466,0.441,0.433,-80.9099,-118.7658,76.0,86.0,Left


#### 2. Standardize for Height and Handedness

We normalize for release height by dividing by player height, since taller players will genreally release the ball at a greater height that shorter players. 

In [24]:
df_std = df.copy()
df_std['Release.Height.Norm'] = df_std['Release.Height'] / df_std['ht']

Left-handed players were transformed into a right-handed reference frame by reflecting spatial coordinates and swapping left/right biomechanical variables, ensuring all ‘dominant-side’ features are consistently aligned across observations.

In [30]:
def standardize_handedness(df):
    df = df_std.copy()
   
    # 1. Identify left-handed players
    left_mask = df['hand'] == 'Left'
    
    # 2. SPATIAL NORMALIZATION
    # Assumes hoop at (0,0)
    
    def reflect_across_shot_axis(x, y):
        norm = np.sqrt(x**2 + y**2)
        if norm == 0:
            return x, y
        
        ux, uy = x / norm, y / norm
        proj = x * ux + y * uy
        
        px, py = proj * ux, proj * uy
        perp_x, perp_y = x - px, y - py
        
        # Reflect across axis
        rx = x - 2 * perp_x
        ry = y - 2 * perp_y
        
        return rx, ry
    
    # Apply to shot location
    coords = df.loc[left_mask, ['Shot.X.Pos', 'Shot.Y.Pos']].values
    reflected = np.array([reflect_across_shot_axis(x, y) for x, y in coords])
    
    df.loc[left_mask, 'Shot.X.Pos'] = reflected[:, 0]
    df.loc[left_mask, 'Shot.Y.Pos'] = reflected[:, 1]
    
    # Global flip (left court --> right court)
    df.loc[left_mask, 'Shot.X.Pos'] *= -1
    
    # Flip left_right indicator if exists
    if 'left_right' in df.columns:
        df.loc[left_mask, 'left_right'] *= -1
    
    # 3. APPLY TO ALL X COORDINATES
    # Automatically detect X-position columns (ball tracking etc.)
    
    x_cols = [col for col in df.columns if re.search(r'(^|_)X($|[A-Z])', col)]
    y_cols = [col for col in df.columns if re.search(r'(^|_)Y($|[A-Z])', col)]
    
    for x_col, y_col in zip(x_cols, y_cols):
        coords = df.loc[left_mask, [x_col, y_col]].values
        
        reflected = np.array([
            reflect_across_shot_axis(x, y) for x, y in coords
        ])
        
        df.loc[left_mask, x_col] = reflected[:, 0]
        df.loc[left_mask, y_col] = reflected[:, 1]
        
        # Global flip
        df.loc[left_mask, x_col] *= -1

    
    # 4. BIOMECHANICAL NORMALIZATION
    # Swap ALL Left/Right columns automatically
    
    left_cols = [col for col in df.columns if 'Left' in col]
    
    for left_col in left_cols:
        right_col = left_col.replace('Left', 'Right')
        
        if right_col in df.columns:
            # Swap values ONLY for left-handed players
            temp = df.loc[left_mask, left_col].copy()
            df.loc[left_mask, left_col] = df.loc[left_mask, right_col]
            df.loc[left_mask, right_col] = temp
    
    
    # 5. Rename to Dominant / Non-Dominant
    
    rename_dict = {}
    for col in df.columns:
        if 'Right' in col:
            rename_dict[col] = col.replace('Right', 'Dominant')
        elif 'Left' in col:
            rename_dict[col] = col.replace('Left', 'NonDominant')
    
    df = df.rename(columns=rename_dict)
    
    
    # 6. Mark standardized handedness
    df['hand_standardized'] = 'Right'
    
    
    return df


# Run pipeline
df_std1 = standardize_handedness(df_std)

print("Full handedness normalization complete.")

Full handedness normalization complete.


#### 3. Random Forest Model to Find Optimal Ball Flight Ranges

This pipeline groups shots by location and type, then trains a Random Forest model within each group to predict Ball Distance from Center. It identifies the “optimal” shots (lowest predicted error) and uses those to compute ideal ranges for key ball-flight predictors, while also extracting the most important variables influencing shot quality in each situation.

In [39]:
PREDICTORS = [
    'Ball_to_Rim_Angle',
    'Ball_Velocity',
    'Release.Height.Norm',
    'Initial.Ball.Angle',
    'Initial.Ball.Velocity',
    'Max.Ball.Arc'
]
TARGET = 'Ball.Distance.from.Center'
PREDICTOR_LABELS = {
    'Ball_to_Rim_Angle': 'Ball to Rim Angle',
    'Ball_Velocity': 'Ball Velocity',
    'Release.Height.Norm': 'Release Height (Norm)',
    'Initial.Ball.Angle': 'Initial Ball Angle',
    'Initial.Ball.Velocity': 'Initial Ball Velocity',
    'Max.Ball.Arc': 'Max Ball Arc'
}
MIN_N = 100
OPTIMAL_PERCENTILE = 25


# Random Forest: BDfC --> Optimal Ranges

from sklearn.ensemble import RandomForestRegressor

results = []

# 1. Build groups
groups = []

for loc in df_std1['Shot.Location'].unique():

    if loc == 'Free Throw':
        subset = df_std1[df_std1['Shot.Location'] == loc].copy()

        if len(subset) >= MIN_N:
            groups.append((loc, 'All', subset))

    else:
        for shot_type in df_std1[df_std1['Shot.Location'] == loc]['Shot.Type'].unique():

            subset = df_std1[
                (df_std1['Shot.Location'] == loc) &
                (df_std1['Shot.Type'] == shot_type)
            ].copy()

            if len(subset) >= MIN_N:
                groups.append((loc, shot_type, subset))


# 2. Loop through groups
for loc, shot_type, subset in groups:

    label = f"{loc} | {shot_type}"
    n = len(subset)
    # Fit Random Forest
    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(subset[PREDICTORS], subset[TARGET])

    # Predict BDfC
    subset = subset.copy()
    subset['predicted_BDfC'] = rf.predict(subset[PREDICTORS])

    # Select optimal shots (lowest 25%)
    threshold = subset['predicted_BDfC'].quantile(OPTIMAL_PERCENTILE / 100)
    optimal = subset[subset['predicted_BDfC'] <= threshold]

    # Feature importance
    importances = pd.Series(
        rf.feature_importances_,
        index=PREDICTORS
    ).sort_values(ascending=False)

    top1 = importances.index[0]
    top2 = importances.index[1]

    # Build output row
    row = {
        "Shot Location": loc,
        "Shot Type": shot_type,
        "n": n,
        "n optimal": len(optimal),
        "Top Predictor": PREDICTOR_LABELS[top1],
        "2nd Predictor": PREDICTOR_LABELS[top2],
    }

    # 10th–90th percentile ranges
    for pred in PREDICTORS:
        row[PREDICTOR_LABELS[pred] + " Range"] = (
            round(optimal[pred].quantile(0.10), 3),
            round(optimal[pred].quantile(0.90), 3)
        )

    results.append(row)

# 3. Final output table
rf_ranges_df = pd.DataFrame(results)

rf_ranges_df

,Shot Location,Shot Type,n,n optimal,Top Predictor,2nd Predictor,Ball to Rim Angle Range,Ball Velocity Range,Release Height (Norm) Range,Initial Ball Angle Range,Initial Ball Velocity Range,Max Ball Arc Range
0,Left Corner Three,Off the Dribble,3996,999,Ball Velocity,Initial Ball Velocity,"(42.516, 48.462)","(1.587, 5.538)","(0.032, 0.04)","(41.09, 48.175)","(7.955, 8.344)","(4.386, 4.95)"
1,Left Corner Three,Catch and Shoot,994,249,Initial Ball Velocity,Ball Velocity,"(43.404, 48.852)","(4.91, 5.503)","(0.033, 0.04)","(41.792, 47.187)","(7.992, 8.301)","(4.461, 4.943)"
2,Right Corner Three,Catch and Shoot,2620,655,Ball Velocity,Initial Ball Velocity,"(42.718, 48.59)","(1.58, 5.504)","(0.028, 0.033)","(44.499, 52.045)","(7.976, 8.364)","(4.288, 4.725)"
3,Right Corner Three,Off the Dribble,2338,585,Ball Velocity,Max Ball Arc,"(42.82, 48.062)","(1.607, 5.534)","(0.028, 0.034)","(43.695, 50.971)","(7.938, 8.329)","(4.228, 4.683)"
4,Middle Three,Catch and Shoot,4274,1069,Initial Ball Velocity,Release Height (Norm),"(41.588, 47.524)","(1.686, 5.924)","(0.032, 0.035)","(42.335, 47.639)","(8.274, 8.673)","(4.458, 4.937)"
5,Middle Three,Off the Dribble,730,183,Initial Ball Velocity,Initial Ball Angle,"(41.284, 47.05)","(1.734, 6.231)","(0.032, 0.036)","(42.382, 48.353)","(8.307, 9.297)","(4.045, 5.19)"
6,Free Throw,All,3967,992,Initial Ball Velocity,Ball Velocity,"(41.021, 48.805)","(1.236, 4.353)","(0.031, 0.033)","(44.856, 51.551)","(6.382, 6.697)","(3.726, 4.136)"
7,Key,Off the Dribble,1301,326,Ball Velocity,Initial Ball Velocity,"(39.79, 49.92)","(1.207, 4.414)","(0.031, 0.033)","(45.917, 52.744)","(6.03, 6.674)","(3.739, 4.087)"
8,Key,Catch and Shoot,519,130,Release Height (Norm),Initial Ball Velocity,"(42.005, 49.732)","(0.869, 4.341)","(0.031, 0.033)","(45.795, 54.25)","(5.419, 6.635)","(3.62, 4.137)"
9,Right Corner Two,Catch and Shoot,296,74,Max Ball Arc,Ball Velocity,"(44.129, 51.784)","(1.256, 5.083)","(0.03, 0.033)","(45.937, 56.494)","(5.207, 8.238)","(3.493, 4.65)"


#### 4. Optimal Ball Flight Ranges in Regression Modeling (Using Biomechanics to Predict Ball Flight)

This pipeline cleans the dataset to include only numeric biomechanical predictors. It then trains separate LightGBM regression models for each ball-flight outcome. For each model, it generates predictions to compute performance metrics (MAE, RMSE, and R²) and extracts feature importance using LightGBM’s gain-based method, which measures how much each biomechanical variable improves prediction accuracy across all tree splits.

In [75]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. DEFINE VARIABLES

BALL_FLIGHT_TARGETS = [
    'Ball_to_Rim_Angle',
    'Ball_Velocity',
    'Release.Height',
    'Initial.Ball.Angle',
    'Initial.Ball.Velocity',
    'Max.Ball.Arc'
]

BIO_KEYWORDS = [
    "ankle",
    "knee",
    "hip",
    "shoulder",
    "elbow",
    "torso",
    "flexion",
    "extension",
    "alignment",
    "rom",
    "velocity",
    "velo",   # captures TorsoVelo, etc.
    "dorsiflexion",
    "plantarflexion",
    "stance",
    "gap"
]

EXCLUDE_KEYWORDS = [
    "ball",
    "time",
    "shot",
    "made",
    "x",
    "y",
    "z",
    "pos",
    "distance"
]

def select_biomechanical_features(df):
    selected = []

    for c in df.columns:
        c_low = c.lower()

        # must NOT contain excluded terms
        if any(ex in c_low for ex in EXCLUDE_KEYWORDS):
            continue

        # MUST contain biomechanical signal
        if any(bio in c_low for bio in BIO_KEYWORDS):
            selected.append(c)

    return selected


# 2. CLEAN DATASET

biomech_df = df_std1.copy()

biomech_features = select_biomechanical_features(biomech_df)

X = biomech_df[biomech_features]

# safety: numeric only
X = X.select_dtypes(include=[np.number])


# 3. STORAGE OBJECTS

models = {}
metrics = {}
feature_importance = {}


# 4. TRAIN MODELS

for target in BALL_FLIGHT_TARGETS:

    print(f"\n==============================")
    print(f"Training model: {target}")
    print(f"==============================")

    # align dataset per target
    data = pd.concat([X, biomech_df[target]], axis=1).dropna()

    X_clean = data[X.columns]
    y_clean = data[target]

    # model
    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        random_state=42
    )

    model.fit(X_clean, y_clean)

    # store model
    models[target] = model


    # 5. PREDICTIONS + METRICS

    preds = model.predict(X_clean)

    mae = mean_absolute_error(y_clean, preds)
    rmse = np.sqrt(mean_squared_error(y_clean, preds))
    r2 = r2_score(y_clean, preds)

    metrics[target] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

    print("\nModel Performance:")
    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2  : {r2:.4f}")


    # 6. FEATURE IMPORTANCE (GAIN)

    fi_gain = pd.Series(
        model.booster_.feature_importance(importance_type='gain'),
        index=X_clean.columns
    ).sort_values(ascending=False)

    feature_importance[target] = fi_gain

    print("\nTop biomechanical drivers (GAIN importance):")
    print(fi_gain.head(10))


Training model: Ball_to_Rim_Angle
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003817 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5610
[LightGBM] [Info] Number of data points in the train set: 25868, number of used features: 22
[LightGBM] [Info] Start training from score 45.192212

Model Performance:
MAE : 1.3391
RMSE: 1.8350
R2  : 0.7000

Top biomechanical drivers (GAIN importance):
TorsoVeloRelease               200713.523218
ankleTotalROMAvg               165091.420699
HipAlignmentRelease            150775.719784
ShoulderAlignmentRelease       128509.527004
KneeTotalROMAvg                128450.476014
ShoulderTotalROMDominant       127252.789370
ElbowTotalROMNonDominant       124866.294346
TorsoVeloHitch                 124782.720271
ElbowTotalROMDominant          119861.143484
ShoulderTotalROMNonDominant    101760.082762
dtype: float64

Training model: Ball_Velocity
[LightGBM] [Info] Aut

#### 5. Run SHAP to Explain Feature Contributions to Predictions